[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/math/g2_rolling/g2_rolling.ipynb)

# Octonionic Rolling

This example plans the motion of a marked ball rolling without slipping or twisting over a fixed sphere three times its radius. Each ball placement combines the contact point on the sphere with the ball's spatial attitude, and rolling along a great circle couples the contact motion to four times as much spatial rotation. At this one-to-three radius ratio, split-octonion multiplication governs reachability directly because two configurations lie one great-circle roll apart whenever their product vanishes.

A configuration is modelled as a scalar-free multivector of space `handedness * contact - contact.dual() * attitude`, pairing the tangent-plane half turn `handedness * contact` in the even grade with the reflected attitude `-contact.dual() * attitude` in the odd grade. Cayley-Dickson doubling across the even and odd grades defines `split_product` inside the geometric algebra of space, while the clifford-conjugate scalar product defines the split-signature bilinear form `pairing`. Left multiplication `split_product(start, State)` forms a linear map whose kernel consists of every configuration one roll from `start`, so orthogonal projection onto that kernel together with one further product finds multi-roll routes algebraically, and commutators of left and right multiplications generate route-preserving symmetries in split G2 ([Baez and Huerta, *G₂ and the Rolling Ball*](https://arxiv.org/html/1205.2447v4)).

In Cayley-Dickson quaternion-pair notation on $\mathbb{O}' = \mathbb{H} \oplus \mathbb{H}\ell$ ($\ell^2 = +1$), elements are formal pairs $(p_1, q_1), (p_2, q_2) \in \mathbb{H} \times \mathbb{H}$ multiplied by $(p_1, q_1)(p_2, q_2) = (p_1 p_2 + \bar{q}_2 q_1,\, q_2 p_1 + q_1 \bar{p}_2)$ under the split quadratic form $|p|^2 - |q|^2$, encoding a unit contact vector $\mathbf{n} \in S^2 \subset \mathrm{Im}\,\mathbb{H}$ and unit quaternion $q \in \mathrm{Sp}(1)$ as $(\mathbf{n},\, \mathbf{n}q) \in \mathbb{P}(\mathrm{Im}\,\mathbb{O}')$. In Gibbs-Heaviside vector kinematics on $S^2 \times \mathrm{SO}(3)$, rolling a ball of radius $r_1 = 1$ on a sphere of radius $r_2 = 3$ without slipping or twisting imposes the nonholonomic velocity constraints $(r_1 + r_2)\dot{\mathbf{n}} = \boldsymbol{\omega} \times \mathbf{n}$ and $\boldsymbol{\omega} \cdot \mathbf{n} = 0$ with $\dot{R} R^\top = [\boldsymbol{\omega}]_\times$, defining the rank-2 Cartan distribution whose $(2, 3, 5)$ bracket growth has symmetry algebra $\mathfrak{g}_2' = \mathfrak{der}(\mathbb{O}')$.

In [ ]:
# In Colab, install numga and its examples from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from numga import NumpyContext, stack
from numga.algebras import VGA3D as ga
from examples.util.animation import show_states
from examples.math.g2_rolling import render

mv = NumpyContext(ga).multivector
# Unit chiral scalar identifying an intrinsic contact direction with the extrinsic plane perpendicular to it.
Handedness = ga.gatype.antipseudoscalar()                            # Antipseudoscalar-
Plane = ga.gatype.vector()                                           # Vector+
Point = ga.gatype.antivector()                                       # Antivector-
Rotor = ga.gatype.rotor()                                            # Rotor+
Octonion = ga.gatype.full()                                          # Full+
State = ga.gatype.nonscalar()                                        # Nonscalar+
StateMap = ga.gatype((State, State))                                 # State+ <- State+
handedness = mv(Handedness, [1])                                     # [] Scalar-
# Bilinear form of the split norm: even contact norm squared minus odd attitude norm squared.
pairing = Octonion.clifford_conjugate().scalar_product(Octonion)     # [] Scalar+ <- (Octonion+, Octonion+)

## Configurations and their product

A ball configuration is modelled as a scalar-free multivector whose even part `handedness * contact` is the half turn in the plane perpendicular to the contact point and whose odd part `-contact.dual() * attitude` combines the ball's attitude with reflection in that plane. Because those two parts have equal Euclidean norm, every configuration has vanishing split norm `pairing(state, state) == 0`. Scaling a configuration leaves the physical state unchanged, and negating it swaps the contact point to its antipode until a continuous hemisphere representative is chosen.

Splitting the octonion product across the even and odd grades requires the contact contribution to carry the same extrinsic orientation as `attitude`, whereas a contact point is intrinsic like every spatial direction. In three dimensions the plane perpendicular to a direction has the same grade as the direction itself, and identifying the two uses the unit intrinsic scalar `handedness`. Reversing `handedness` negates the odd part of every configuration while preserving `split_product`, so reachability by rolling is independent of that convention.

In [ ]:
def split_product(first: Octonion, second: Octonion) -> Octonion:
    """Cayley-Dickson doubling across the even and odd grades multiplies two space multivectors (`[...] Octonion+`) as split octonions."""
    first_even, first_odd = first.restrict.even(), first.restrict.odd()        # [...] Even+, Odd+
    second_even, second_odd = second.restrict.even(), second.restrict.odd()    # [...] Even+, Odd+
    # Odd parts multiply in swapped order, with reversal acting as quaternion conjugation.
    return (first_even * second_even + second_odd * first_odd.reverse()
            + first_even.reverse() * second_odd + second_even * first_odd)

In [ ]:
def configuration(contact: Point, attitude: Rotor) -> State:
    """A unit contact point (`[...] Point-`) and ball attitude (`[...] Rotor+`) combine into one null configuration (`[...] State+`)."""
    # Convert the intrinsic contact point into an extrinsic half-turn bivector in its tangent plane,
    # and form the central reflection plane dual to the contact point.
    half_turn = handedness * contact                                 # [...] Bivector+
    mirror = contact.dual()                                          # [...] Plane+
    return half_turn - mirror * attitude


def contact_and_attitude(state: State) -> tuple[Point, Rotor]:
    """Unit-contact normalization recovers the contact point (`[...] Point-`) and ball attitude (`[...] Rotor+`) from a configuration (`[...] State+`)."""
    unit_state = state / state.restrict[2].norm()                    # [...] State+
    contact = unit_state.restrict[2] / handedness                    # [...] Point-
    mirror = contact.dual()                                          # [...] Plane+
    return contact, -mirror.inverse() * unit_state.restrict.odd()

## Two-roll waypoint from a product

Two configurations whose product vanishes lie on one great-circle roll, parameterized as a straight line in `State` once continuous representatives on one side of a central plane are selected by `hemisphere_lift`. When two configurations instead satisfy `pairing(start, finish) == 0` with a nonzero product, `split_product(start, finish)` is the unique intermediate configuration joining them in two rolls.

In [ ]:
attitude_angle = -2.4
sphere_radius = 3
ball_radius = 1
n_samples = 120
n_steps = 41
n_hold_frames = 13
duration_ms = 40
elevation = 35
azimuth = -25
size = 520

start_contact = mv.z.dual()                                          # [] Point-
start_attitude = mv.rotor()                                          # [] Rotor+
two_roll_contact = mv.x.dual()                                       # [] Point-
target_attitude = (mv.xy * (-attitude_angle / 2)).exp()              # [] Rotor+
# Central plane on whose positive side the lifted contact points lie.
preferred_side = mv.x                                                # [] Plane+

In [ ]:
def hemisphere_lift(state: State, hemisphere: Plane) -> State:
    """Projective sign selection chooses the unit-contact representative (`[...] State+`) whose contact lies on the positive side of `hemisphere`."""
    contact = state.restrict[2] / handedness                         # [...] Point-
    signed_state = state * (hemisphere & contact)                    # [...] State+
    return signed_state / signed_state.restrict[2].norm()


start = configuration(start_contact, start_attitude)                 # [] State+
two_roll_finish = configuration(two_roll_contact, target_attitude)   # [] State+
# Orthogonal endpoints in the split pairing meet at the waypoint given by their product.
waypoint = hemisphere_lift(split_product(start, two_roll_finish).cast(State), preferred_side)   # [] State+
two_roll_stops = stack([start, waypoint, two_roll_finish])           # [3] State+

In [ ]:
def contact_arcs(stops: State, n_samples: int) -> tuple[Point, ...]:
    """Linear interpolation between consecutive `stops` samples `n_samples` contact points (`[n_samples] Point-`) along each great-circle roll."""
    fractions = np.linspace(0, 1, n_samples)
    states = stops[:-1, None] * (1 - fractions) + stops[1:, None] * fractions   # [..., n_samples] State+
    contacts, _ = contact_and_attitude(states)                                  # [..., n_samples] Point-, Rotor+
    return tuple(contacts)


def route_motion(stops: State, n_steps: int, n_hold_frames: int) -> Iterator[tuple[Point, Rotor]]:
    """Yields `(Point-, Rotor+)` ball placements across `n_steps` interpolation steps per roll of `stops`, holding each endpoint for `n_hold_frames` frames."""
    fractions = np.linspace(0, 1, n_steps)
    for _ in range(n_hold_frames):
        yield contact_and_attitude(stops[0])
    for first, second in zip(stops[:-1], stops[1:]):
        for fraction in fractions:
            yield contact_and_attitude(first * (1 - fraction) + second * fraction)
    for _ in range(n_hold_frames):
        yield contact_and_attitude(stops[-1])


two_roll_placements = show_states(
    route_motion(two_roll_stops, n_steps, n_hold_frames),
    render.route_frames, contact_arcs(two_roll_stops, n_samples),
    sphere_radius, ball_radius, elevation, azimuth, size,
    name="g2_two_rolls", duration_ms=duration_ms,
);

The blue and orange great-circle arcs trace the two contact rolls on the fixed sphere, while the coloured octant seams on the small ball show its spatial attitude turning four times as fast as the contact angle without slipping or twisting.

## Three-roll route from a kernel

Returning to the starting contact point with a new attitude gives `pairing(start, finish) != 0`, so at least three rolls are required. Orthogonal projection onto the kernel of `split_product(start, State)` isolates every configuration reachable in one roll, and removing the component that pairs with `finish` leaves first waypoints from which `split_product(first, finish)` completes the remaining two rolls.

In [ ]:
def annihilator(state: State) -> StateMap:
    """Orthogonal projection onto the left-multiplication kernel isolates the configurations (`State+ <- State+`) one roll away from `state`."""
    left_action = split_product(state, State)                        # [] Octonion+ <- State+
    return State - left_action.lstsq(left_action)


def three_roll_waypoints(start: State, finish: State, preferred_contact: Point) -> tuple[State, State]:
    """Kernel projection and one split-octonion product determine two intermediate waypoints (`State+`) between endpoints three rolls apart."""
    first_roll_space = annihilator(start)                            # [] State+ <- State+
    # Vanishing split pairing with the destination puts a first waypoint within two rolls of `finish`.
    finish_pairing = pairing(finish, first_roll_space)               # [] Scalar+ <- State+
    preference = handedness * preferred_contact                      # [] Bivector+
    first = first_roll_space(preference - finish_pairing.lstsq(finish_pairing(preference)))   # [] State+
    # The split-octonion product supplies the unique waypoint joining a two-roll pair.
    second = split_product(first, finish).cast(State)                # [] State+
    return first / first.restrict[2].norm(), second / second.restrict[2].norm()


three_roll_finish = configuration(start_contact, target_attitude)    # [] State+
first_waypoint, second_waypoint = three_roll_waypoints(start, three_roll_finish, preferred_side.dual())   # [] State+, State+
waypoints = hemisphere_lift(stack([first_waypoint, second_waypoint]), preferred_side)                     # [2] State+
three_roll_stops = stack([start, *waypoints, three_roll_finish])     # [4] State+

three_roll_placements = show_states(
    route_motion(three_roll_stops, n_steps, n_hold_frames),
    render.route_frames, contact_arcs(three_roll_stops, n_samples),
    sphere_radius, ball_radius, elevation, azimuth, size,
    name="g2_three_rolls", duration_ms=duration_ms,
);

The ball traverses the blue, orange, and purple great-circle arcs and returns to the north-pole contact rotated in the horizontal plane by `attitude_angle` radians, with both intermediate waypoints fixed directly by the kernel projection toward `preferred_side` and one split-octonion product.

## Split-G2 symmetry of a route

Invertible linear maps that preserve `split_product` form the exceptional group split G2, and because they preserve every vanishing product between consecutive stops they carry any valid rolling route to another valid rolling route. Commutators of left and right multiplications define an infinitesimal derivation `derivation(start, plane)` that annihilates `start` and has vanishing cube, so its quadratic polynomial gives an exact finite symmetry fixing the initial configuration while deforming the rest of the route.

In [ ]:
symmetry_amount = 0.12
panel_size = 370

In [ ]:
def derivation(first: Octonion, second: Octonion) -> StateMap:
    """Commutators of left and right split-octonion multiplications form an infinitesimal automorphism (`State+ <- State+`) of the product."""
    left_first = split_product(first, Octonion)                      # [] Octonion+ <- Octonion+
    left_second = split_product(second, Octonion)                    # [] Octonion+ <- Octonion+
    right_first = split_product(Octonion, first)                     # [] Octonion+ <- Octonion+
    right_second = split_product(Octonion, second)                   # [] Octonion+ <- Octonion+
    # Alternating left and right multiplication commutators cancel the non-associator on every product.
    octonion_action = (left_first(left_second) - left_second(left_first)
                       + left_first(right_second) - right_second(left_first)
                       + right_first(right_second) - right_second(right_first))   # [] Octonion+ <- Octonion+
    return octonion_action(State).cast(State)


# The third composition of this derivation vanishes, so its exponential truncates at degree two.
generator = derivation(start, mv.x)                                  # [] State+ <- State+
symmetry = State + symmetry_amount * generator + symmetry_amount**2 / 2 * generator(generator)   # [] State+ <- State+
transformed_stops = symmetry(three_roll_stops)                       # [4] State+

paired_placements = show_states(
    zip(
        route_motion(three_roll_stops, n_steps, n_hold_frames),
        route_motion(transformed_stops, n_steps, n_hold_frames),
    ),
    render.comparison_frames,
    contact_arcs(three_roll_stops, n_samples), contact_arcs(transformed_stops, n_samples),
    sphere_radius, ball_radius, elevation, azimuth, panel_size,
    name="g2_route_symmetry", duration_ms=duration_ms,
);

Both spheres start from the same north-pole placement, yet the split-G2 image on the right bends the three great-circle arcs and shifts the final contact point while keeping every leg a genuine no-slip, no-twist roll.

In [ ]:
# checks
# Every stop lies on the split-octonion null cone, and consecutive stops multiply to zero.
for route_stops in (two_roll_stops, three_roll_stops, transformed_stops):
    assert (pairing(route_stops, route_stops).abs() < 1e-10).all()
    assert (split_product(route_stops[:-1], route_stops[1:]).scalar_norm_squared() < 1e-10).all()

# The split-G2 generator fixes `start`, has vanishing cube, and preserves physical rolling on every streamed step.
assert (generator(start).scalar_norm_squared() < 1e-12).all()
third_power = generator(generator(generator))                        # [] State+ <- State+
assert (third_power.adjoint()(third_power).trace() < 1e-12).all()
assert ((two_roll_placements[-1][0] - two_roll_contact).scalar_norm_squared() < 1e-12).all()
assert ((three_roll_placements[-1][0] - start_contact).scalar_norm_squared() < 1e-12).all()

for placements in (
    two_roll_placements,
    three_roll_placements,
    [pair[0] for pair in paired_placements],
    [pair[1] for pair in paired_placements],
):
    contacts = stack([contact for contact, _ in placements])         # [...] Point-
    attitudes = stack([attitude for _, attitude in placements])      # [...] Rotor+
    orbit_turn = (1 + contacts[1:].dual() * contacts[:-1].dual()).normalized()   # [...] Rotor+
    ball_turn = orbit_turn * orbit_turn * orbit_turn * orbit_turn    # [...] Rotor+
    assert (((ball_turn * attitudes[:-1]) - attitudes[1:]).scalar_norm_squared() < 1e-10).all()